# 01 — Data and Coverage QC

## Groundwater–InSAR Reproducible Workflow

This notebook reconstructs and verifies the three foundational datasets used in the groundwater–InSAR analysis:

1. **Stage 01 — Monthly groundwater observations**
2. **Stage 02 — InSAR epoch values sampled at piezometers**
3. **Stage 03 — Monthly InSAR aggregation**

The calculations are performed by version-controlled Python modules in `src/groundwater_insar/`.  
The notebook itself is used for scientific documentation, quality control, visualization, and reproducibility.

### Sign conventions

- Positive `GW_Decline_m` = groundwater-level decline.
- Positive InSAR displacement = subsidence.
- `InSAR_Increment_mm = current monthly cumulative subsidence − previous monthly cumulative subsidence`.

No original terminal-analysis CSV is overwritten. Reconstructed products are written to the dedicated `regenerated` directory and compared against the frozen terminal baseline.

In [ ]:
from pathlib import Path
import json
import platform
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from groundwater_insar.config import load_config
from groundwater_insar.stage01_groundwater import reconstruct_stage01
from groundwater_insar.stage02_insar_epochs import reconstruct_stage02
from groundwater_insar.stage03_insar_monthly import reconstruct_stage03

cfg = load_config()

print("Python      :", sys.version.split()[0])
print("Platform    :", platform.platform())
print("Repository  :", cfg["paths"]["repository_root"])
print("Workspace   :", cfg["paths"]["workspace_root"])
print("Analysis    :", cfg["paths"]["analysis_root"])
print("Regenerated :", cfg["paths"]["regenerated_root"])

## 1. Reproducibility and provenance

The notebook records the current Git branch and commit before the scientific reconstruction.  
The external notebook runner additionally records source hashes, configuration hashes, environment information, execution status, and Git state in the run manifest.

In [ ]:
repo = Path(cfg["paths"]["repository_root"])

def git(*args):
    return subprocess.check_output(
        ["git", "-C", str(repo), *args],
        text=True
    ).strip()

print("Git branch :", git("branch", "--show-current"))
print("Git commit :", git("rev-parse", "HEAD"))

status = git("status", "--porcelain")
print("Git dirty  :", bool(status))

if status:
    print("\nCurrent uncommitted files:")
    print(status)

## 2. Stage 01 — Monthly groundwater reconstruction

The raw monthly groundwater table contains the complete piezometer archive.  
The official analysis subset is defined centrally in the project configuration.

For each official piezometer, the monthly series is converted to long format and the groundwater decline is calculated as:

**GW decline = previous monthly head − current monthly head**

The previous-month chain begins at the analysis-period boundary; therefore, the first record of each piezometer has no previous-month value.

In [ ]:
stage01, stage01_out, stage01_ref = reconstruct_stage01(cfg)

print("Generated :", stage01_out)
print("Reference :", stage01_ref)
print("Rows      :", len(stage01))
print("Wells     :", stage01["Piezometer"].nunique())

display(stage01.head())

In [ ]:
stage01_qc = pd.DataFrame({
    "Metric": [
        "Rows",
        "Official piezometers",
        "Missing Head_m",
        "Missing Prev_Head_m",
        "Missing GW_Decline_m",
        "Missing Prev_MonthIndex",
    ],
    "Value": [
        len(stage01),
        stage01["Piezometer"].nunique(),
        stage01["Head_m"].isna().sum(),
        stage01["Prev_Head_m"].isna().sum(),
        stage01["GW_Decline_m"].isna().sum(),
        stage01["Prev_MonthIndex"].isna().sum(),
    ],
})

display(stage01_qc)

## 3. Stage 02 — InSAR epoch sampling at piezometers

All official piezometers are matched to their UTM coordinates and sampled from every cumulative SBAS-InSAR epoch raster.

Important data-validity rule recovered during reproducibility testing:

- ordinary missing raster samples are stored as `NaN`;
- the float32 sentinel near `−3.402823 × 10³⁸` is also treated as NoData;
- a well–epoch row is retained even when its InSAR value is missing.

This preserves the complete temporal sampling structure instead of deleting invalid observations.

In [ ]:
stage02, stage02_out, stage02_ref, coordinates, epochs = reconstruct_stage02(cfg)

print("Generated       :", stage02_out)
print("Reference       :", stage02_ref)
print("Official wells  :", len(coordinates))
print("Epochs          :", len(epochs))
print("Rows            :", len(stage02))
print("Valid InSAR     :", stage02["InSAR_mm"].notna().sum())
print("Missing InSAR   :", stage02["InSAR_mm"].isna().sum())

display(stage02.head())

In [ ]:
epoch_inventory = (
    stage02[
        ["StackYear", "AcquisitionDate", "Raster"]
    ]
    .drop_duplicates()
    .groupby("StackYear")
    .size()
    .rename("N_Epochs")
    .reset_index()
)

display(epoch_inventory)

print(
    "Total epochs:",
    int(epoch_inventory["N_Epochs"].sum())
)

In [ ]:
well_coverage = (
    stage02
    .groupby("Piezometer")
    .agg(
        Total_Epochs=("InSAR_mm", "size"),
        Valid_Epochs=("InSAR_mm", "count"),
    )
)

well_coverage["Missing_Epochs"] = (
    well_coverage["Total_Epochs"]
    - well_coverage["Valid_Epochs"]
)

well_coverage["Valid_Percent"] = (
    100
    * well_coverage["Valid_Epochs"]
    / well_coverage["Total_Epochs"]
)

display(
    well_coverage
    .sort_values(
        ["Valid_Epochs", "Piezometer"],
        ascending=[True, True]
    )
)

## 4. Stage 03 — Monthly InSAR aggregation

Epoch observations are aggregated within each:

`StackYear × Piezometer × Jalali month`

The monthly InSAR value is the **mean of valid epoch observations**.

`N_Epochs` counts only valid InSAR values, while first and last acquisition dates preserve the acquisition-window information.

Previous-month variables are calculated independently within each `StackYear × Piezometer` series, so the temporal chain resets at each SBAS stack boundary.

In [ ]:
stage03, stage03_out, stage03_ref = reconstruct_stage03(cfg)

print("Generated :", stage03_out)
print("Reference :", stage03_ref)
print("Rows      :", len(stage03))
print("Wells     :", stage03["Piezometer"].nunique())
print("Stacks    :", stage03["StackYear"].nunique())

print(
    "Missing monthly InSAR :",
    stage03["InSAR_Monthly_mm"].isna().sum()
)

print(
    "Missing increments    :",
    stage03["InSAR_Increment_mm"].isna().sum()
)

display(stage03.head())

In [ ]:
monthly_epoch_distribution = (
    stage03["N_Epochs"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("N_Epochs")
    .reset_index(name="Monthly_Rows")
)

display(monthly_epoch_distribution)

assert (
    int(stage03["N_Epochs"].sum())
    ==
    int(stage02["InSAR_mm"].notna().sum())
)

print("Valid-epoch conservation: PASS")

In [ ]:
stack_coverage = (
    stage03
    .groupby("StackYear")
    .agg(
        Monthly_Rows=("InSAR_Monthly_mm", "size"),
        Valid_Monthly=("InSAR_Monthly_mm", "count"),
        Valid_Increments=("InSAR_Increment_mm", "count"),
    )
)

stack_coverage["Missing_Monthly"] = (
    stack_coverage["Monthly_Rows"]
    - stack_coverage["Valid_Monthly"]
)

stack_coverage["Monthly_Coverage_Percent"] = (
    100
    * stack_coverage["Valid_Monthly"]
    / stack_coverage["Monthly_Rows"]
)

display(stack_coverage)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

ax.bar(
    stack_coverage.index.astype(str),
    stack_coverage["Monthly_Coverage_Percent"]
)

ax.set_xlabel("SBAS Stack Year")
ax.set_ylabel("Valid monthly InSAR coverage (%)")
ax.set_title("Monthly InSAR coverage by SBAS stack")
ax.set_ylim(0, 105)

plt.tight_layout()
plt.show()

## 5. Frozen-baseline regression tests

Each reconstructed stage is compared with the frozen terminal-analysis baseline.

A `SEMANTIC_MATCH` or `BYTE_IDENTICAL` result is accepted.  
Any scientific value mismatch causes the notebook workflow to fail.

In [ ]:
regenerated_root = Path(
    cfg["paths"]["regenerated_root"]
)

regression_runs = [
    (
        1,
        regenerated_root / "stage01",
        "01_Stage01_Groundwater_Regression.csv",
    ),
    (
        2,
        regenerated_root / "stage02",
        "02_Stage02_InSAR_Epochs_Regression.csv",
    ),
    (
        3,
        regenerated_root / "stage03",
        "03_Stage03_InSAR_Monthly_Regression.csv",
    ),
]

for stage, candidate_dir, report_name in regression_runs:

    print("\n" + "=" * 90)
    print(f"REGRESSION — STAGE {stage:02d}")
    print("=" * 90)

    cmd = [
        sys.executable,
        "-m",
        "groundwater_insar.regression",
        "--candidate-dir",
        str(candidate_dir),
        "--stage",
        f"{stage:02d}",
        "--report-name",
        report_name,
    ]

    result = subprocess.run(
        cmd,
        text=True,
        capture_output=True
    )

    print(result.stdout)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(
            f"Stage {stage:02d} regression failed."
        )

print("\nSTAGES 01–03 REGRESSION: PASS")

## 6. Notebook QC summary

Passing this notebook establishes that the groundwater observations, InSAR epoch sampling, and monthly InSAR aggregation can be reconstructed from source data and reproduce the frozen terminal-analysis results.

This notebook does **not** yet perform groundwater–InSAR pairing, lag analysis, or causal interpretation. Those operations belong to subsequent workflow stages.

In [ ]:
final_qc = pd.DataFrame({
    "Stage": [
        "01 — Groundwater monthly",
        "02 — InSAR epochs",
        "03 — InSAR monthly",
    ],
    "Rows": [
        len(stage01),
        len(stage02),
        len(stage03),
    ],
    "Status": [
        "PASS",
        "PASS",
        "PASS",
    ],
})

display(final_qc)

print()
print("01_DATA_AND_COVERAGE_QC NOTEBOOK: PASS")